In [0]:
# Configuración — celda completa, no modificar
CATALOGO      = "dbassociate"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
VOL_LANDING   = "/Volumes/dbassociate/default/vol_landing/session_07"
CSV_MOV       = f"{VOL_LANDING}/movimientos_suscripcion.csv"

TABLA_BRONZE  = f"{CATALOGO}.{SCHEMA_BRONZE}.movimientos"
TABLA_SILVER  = f"{CATALOGO}.{SCHEMA_SILVER}.movimientos"

try:
    dbutils.fs.ls(CSV_MOV)
    print(f"OK: {CSV_MOV}")
except Exception:
    raise FileNotFoundError(
        f"Subir movimientos_suscripcion.csv a {VOL_LANDING} antes de continuar."
    )

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, DecimalType

In [0]:
schema = StructType([
    StructField("movimiento_id", StringType(), True),
    StructField("suscripcion_id", StringType(), True),
    StructField("tipo_movimiento", StringType(), True),
    StructField("plan_anterior", StringType(), True),
    StructField("plan_nuevo", StringType(), True),
    StructField("monto_anterior", DecimalType(10,2), True),
    StructField("monto_nuevo", DecimalType(10,2), True),
    StructField("fecha", DateType(), True),
    StructField("motivo", StringType(), True)
])

df_subs = (spark.read
    .option("header", "true")
    .schema(schema)
    .csv(CSV_MOV)
    )

In [0]:
df_subs_batch_01 = df_subs.filter(
    "tipo_movimiento in ('Alta', 'Upgrade') and movimiento_id <= 'MOV-047'"
)

df_subs_batch_02 = df_subs.filter(
    "movimiento_id > 'MOV-047'"
)

In [0]:
%sql

CREATE TABLE dbassociate.bronze.movimiento (
  movimiento_id STRING, 
  suscripcion_id STRING, 
  tipo_movimiento STRING,
  plan_anterior STRING,
  plan_nuevo STRING,
  monto_anterior DECIMAL(10,2),
  monto_nuevo DECIMAL(10,2),
  fecha DATE,
  motivo STRING
)
USING delta
PARTITIONED BY (tipo_movimiento)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
    )

In [0]:
from pyspark.sql import functions as F

In [0]:
(df_subs_batch_01
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("dbassociate.bronze.movimiento")
)

In [0]:
%sql
-- DROP TABLE dbassociate.bronze.movimiento
-- DROP TABLE dbassociate.silver.movimiento

--RESTORE TABLE dbassociate.bronze.movimiento TO VERSION AS OF 0;

-- SELECT _commit_version, _change_type, count(*) as records
-- from table_changes('dbassociate.bronze.movimiento', 1)
-- GROUP BY _commit_version, _change_type

In [0]:
df_silver = (
    spark.read
    .option("readChangeFeed", "true")
    .option("startingVersion", "0")
    .table("dbassociate.bronze.movimiento")
    .drop("_change_type", "_commit_version", "_commit_timestamp")
)

In [0]:
#Carga inicial
(
    df_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dbassociate.silver.movimiento")
)

In [0]:
%sql

SELECT *
FROM dbassociate.silver.movimiento

In [0]:
%sql

-- DROP TABLE dbassociate.bronze.log_runs

-- CREATE TABLE IF NOT EXISTS dbassociate.bronze.log_runs AS
-- SELECT
--   "Subscripciones" as proceso,
--   CAST(current_timestamp() as DATE) as fecha,
--   max(_commit_version) as version_actual
-- from table_changes('dbassociate.bronze.movimiento', 1)



SELECT *
FROM dbassociate.bronze.log_runs

In [0]:
starting_version = (
    spark.read
        .table("dbassociate.bronze.log_runs")
        .filter("proceso = 'Subscripciones'")
        .select(F.col("version_actual"))
        .collect()[0][0]
)

In [0]:
df_staging_silver = (
    spark.read
    .option("readChangeFeed", "true")
    .option("startingVersion", starting_version)
    .table("dbassociate.bronze.movimiento")
    .filter("_change_type in ('insert', 'update_postimage')")
)

df_staging_silver.createOrReplaceTempView("df_staging_silver")

In [0]:
%sql

MERGE INTO dbassociate.silver.movimiento as t
USING df_staging_silver as s
ON t.movimiento_id = s.movimiento_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql

DESCRIBE HISTORY dbassociate.silver.movimiento

In [0]:
%sql
SELECT *
FROM dbassociate.silver.movimiento
LIMIT 2

In [0]:
df_subs_batch_02.createOrReplaceTempView("df_subs_batch_02")

(
    df_subs_batch_02.write
        .format("delta")
        .mode("append")
        .saveAsTable("dbassociate.bronze.movimiento")
)

In [0]:
spark.sql(f"""
MERGE INTO dbassociate.bronze.log_runs as t
USING (
    SELECT
        "Subscripciones" as proceso,
        CAST(current_timestamp() as DATE) as fecha,
        max(_commit_version) as version_actual
    FROM table_changes('dbassociate.bronze.movimiento', {starting_version})
) as s
ON t.proceso = s.proceso
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED
    THEN INSERT *
    """)

In [0]:
%sql
SELECT * FROM dbassociate.bronze.log_runs

In [0]:
max_version = (
    spark.read
        .table("dbassociate.bronze.log_runs")
        .filter("proceso = 'Subscripciones'")
        .select(F.col("version_actual"))
        .collect()[0][0]
)

In [0]:
df_staging_silver = (
    spark.read
    .option("readChangeFeed", "true")
    .option("startingVersion", max_version)
    .table("dbassociate.bronze.movimiento")
    .filter("_change_type in ('insert', 'update_postimage')")
)

In [0]:
display(df_staging_silver)

In [0]:
%sql

MERGE INTO dbassociate.silver.movimiento as t
USING df_staging_silver as s
ON t.movimiento_id = s.movimiento_id
WHEN MATCHED AND _change_type = 'update_postimage' THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql

DESCRIBE HISTORY dbassociate.silver.movimiento

In [0]:
TABLA_BRONZE = "dbassociate.bronze.movimiento"
TABLA_SILVER = "dbassociate.silver.movimiento"

In [0]:
bronze_count = spark.table(TABLA_BRONZE).count()
silver_count = spark.table(TABLA_SILVER).count()

print(f"Registros en Bronze: {bronze_count}")
print(f"Registros en Silver: {silver_count}")
print(f"Diferencia: {bronze_count - silver_count}")

if bronze_count == silver_count:
    print("CORRECTO: Bronze y Silver tienen el mismo número de registros.")
else:
    print("REVISAR: El número de registros no coincide.")

print("\n=== Distribución por tipo de movimiento ===")
print("Bronze:")
display(spark.table(TABLA_BRONZE).groupBy("tipo_movimiento").count().orderBy("tipo_movimiento"))

print("Silver:")
display(spark.table(TABLA_SILVER).groupBy("tipo_movimiento").count().orderBy("tipo_movimiento"))

print("\n=== Verificación de duplicados en Silver ===")
duplicados = spark.table(TABLA_SILVER).groupBy("movimiento_id").count().filter("count > 1")
dup_count = duplicados.count()
if dup_count == 0:
    print("CORRECTO: No hay duplicados en Silver.")
else:
    print(f"REVISAR: {dup_count} movimiento_id con duplicados en Silver:")
    display(duplicados)